# 04 — Modo directo: Adquisición de señales y E/S

Este notebook cubre las capacidades extendidas de E/S del CompactStat en modo directo:

- **Trazas de alta velocidad** — adquirir hasta 256 muestras de corriente/potencial a tasas de 10 µs a 20 ms
- **Puertos DAC/ADC externos** — emitir voltajes analógicos y leer sensores externos
- **E/S digital** — controlar líneas de salida digital y leer líneas de entrada mediante bitmasks
- **Control de señal AC** — establecer frecuencia y amplitud para experimentos de impedancia
- **Canal MUX** — conmutar canales del multiplexor

### Requisitos previos
- Driver abierto, dispositivo conectado (ver `02_device_and_instance_management.ipynb`)
- Configuración de celda establecida (ver `03_direct_mode_basics.ipynb`)
- Referencia de manejo de errores: `01_getting_started.ipynb` — Sección 4

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from pyvium import Pyvium

Pyvium.open_driver()
Pyvium.connect_device()
Pyvium.set_connection_mode(1)  # EStat 4EL
Pyvium.set_current_range(2)    # 100 µA
Pyvium.set_filter(2)           # 10 kHz — necesario para trazas rápidas
print("Ready")

## 1. Traza de corriente de alta velocidad

> ⚠️ **No funciona actualmente.** `get_current_trace` llama a la función DLL `IV_getcurrenttrace`, que bloquea el proceso host independientemente de los parámetros o el estado de la celda. La causa raíz está bajo investigación con el soporte de Ivium. Las celdas a continuación se mantienen como referencia y para pruebas futuras una vez que el problema se resuelva.

`get_current_trace(n_points, interval)` captura una ráfaga de muestras de corriente.

- `n_points`: número de muestras (máx. 256)
- `interval`: tiempo entre muestras en segundos (10 µs a 20 ms)

Devuelve una lista de valores de corriente en Amperios.

> **Nota sobre el modo HiSpeed:** Cuando `interval` está por debajo de 2 ms, IviumSoft entra automáticamente en **modo HiSpeed** — los datos se almacenan en el búfer interno del potenciostato durante la adquisición y se transfieren al PC solo después de que finaliza la ráfaga. En este modo el tamaño máximo del búfer es de **8192 puntos** por adquisición (32 768 para CV). Intentar más de 8192 puntos en una sola ráfaga HiSpeed será truncado. Para intervalos ≥ 2 ms, los datos se transmiten en tiempo real sin límite de búfer.

In [ ]:
N_POINTS = 256
INTERVAL = 0.001  # 1 ms entre muestras → 256 ms de ráfaga total

Pyvium.set_potential(0.1)
Pyvium.set_cell_on()
time.sleep(0.05)  # asentamiento

current_trace = Pyvium.get_current_trace(N_POINTS, INTERVAL)
print(f"Acquired {len(current_trace)} samples")
print(f"Mean current: {np.mean(current_trace):.3e} A")
print(f"Std dev     : {np.std(current_trace):.3e} A")

times = [i * INTERVAL * 1000 for i in range(len(current_trace))]  # ms
currents_uA = [c * 1e6 for c in current_trace]  # µA

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(times, currents_uA, lw=0.8)
ax.set_xlabel("Time (ms)")
ax.set_ylabel("Current (µA)")
ax.set_title(f"Current Trace — {N_POINTS} points at {INTERVAL*1000:.1f} ms intervals")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Traza de potencial de alta velocidad

> ⚠️ **No funciona actualmente.** Mismo problema que la Sección 1 — `IV_getpotentialtrace` bloquea el proceso host. Se mantiene como referencia.

Misma API que la traza de corriente pero lee el potencial. Útil para verificar la respuesta de la celda ante un escalón de corriente.

In [ ]:
potential_trace = Pyvium.get_potential_trace(N_POINTS, INTERVAL)

times_ms   = [i * INTERVAL * 1000 for i in range(len(potential_trace))]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(times_ms, potential_trace, 'r-', lw=0.8)
ax.set_xlabel("Time (ms)")
ax.set_ylabel("Potential (V)")
ax.set_title("Potential Trace")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

Pyvium.set_cell_off()

## 3. Comparar trazas de corriente y potencial

> ⚠️ **No funciona actualmente.** Ambas funciones de traza bloquean el proceso host. Se mantiene como referencia.

Adquirir ambas señales simultáneamente (dos ráfagas separadas en las mismas condiciones) para una visión completa.

In [ ]:
Pyvium.set_potential(0.2)
Pyvium.set_cell_on()
time.sleep(0.05)

I_trace = Pyvium.get_current_trace(128, 0.0005)   # 128 pts, 0.5 ms intervalo
E_trace = Pyvium.get_potential_trace(128, 0.0005)

Pyvium.set_cell_off()

t_ms = [i * 0.5 for i in range(128)]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
ax1.plot(t_ms, [c * 1e6 for c in I_trace], 'b-', lw=0.8)
ax1.set_ylabel("Current (µA)")
ax1.grid(True, alpha=0.3)

ax2.plot(t_ms, E_trace, 'r-', lw=0.8)
ax2.set_xlabel("Time (ms)")
ax2.set_ylabel("Potential (V)")
ax2.grid(True, alpha=0.3)

fig.suptitle("Current and Potential Traces")
plt.tight_layout()
plt.show()

## 4. Salida DAC externa

El CompactStat tiene dos canales DAC externos (0 y 1) en el puerto externo. Úsalos para emitir señales de control analógicas a equipos externos (p.ej., activar una lámpara, controlar una válvula, sincronizar con otro instrumento).

Rango de salida: típicamente ±10 V (consulta la especificación de tu dispositivo).

In [ ]:
# Establecer canal DAC 0 a 1.5 V
Pyvium.set_dac(0, 1.5)
print("DAC channel 0 set to 1.5 V")

# Establecer canal DAC 1 a -0.5 V
Pyvium.set_dac(1, -0.5)
print("DAC channel 1 set to -0.5 V")

# Restablecer ambos a 0 V cuando se termine
Pyvium.set_dac(0, 0.0)
Pyvium.set_dac(1, 0.0)
print("DAC channels reset to 0 V")

## 5. Entrada ADC externa

`get_adc(channel)` lee el voltaje en uno de los 8 canales de entrada ADC externos (0–7). Útil para leer sensores externos (temperatura, pH, caudal, etc.) en sincronía con una medición electroquímica.

In [ ]:
for channel in range(4):  # leer canales 0–3
    voltage = Pyvium.get_adc(channel)
    print(f"ADC channel {channel}: {voltage:.4f} V")

## 6. Salida digital

`set_digital_output(bitmask)` controla las líneas de salida digital. Cada bit en el entero mapea a una línea de salida digital.

```
bitmask = 0b00000101  →  líneas 0 y 2 en ALTO, otras en BAJO
```

In [ ]:
# Poner línea de salida digital 0 en ALTO
Pyvium.set_digital_output(0b00000001)
print(f"Digital out: 0b{0b00000001:08b}  (línea 0 en ALTO)")
time.sleep(0.5)

# Poner líneas 0 y 2 en ALTO
Pyvium.set_digital_output(0b00000101)
print(f"Digital out: 0b{0b00000101:08b}  (líneas 0 y 2 en ALTO)")
time.sleep(0.5)

# Todas las líneas en BAJO
Pyvium.set_digital_output(0b00000000)
print("Digital out: all lines LOW")

## 7. Entrada digital

`get_digital_input()` devuelve un bitmask del estado actual de las líneas de entrada digital.

In [ ]:
digin = Pyvium.get_digital_input()
print(f"Digital input bitmask : {digin} (0b{digin:08b})")

# Decodificar líneas individuales
for line in range(8):
    state = "HIGH" if digin & (1 << line) else "LOW"
    print(f"  Line {line}: {state}")

## 8. Control de señal AC (para EIS manual)

`set_ac_frequency()` y `set_ac_amplitude()` configuran la señal de perturbación AC aplicada a la celda. En modo directo esto permite aplicar manualmente una señal AC a una frecuencia fija — útil para mediciones de impedancia de una sola frecuencia o para verificar la configuración antes de ejecutar un método EIS completo.

Para barridos EIS automatizados, usa `start_method()` con un archivo de método EIS en su lugar.

In [ ]:
# Configurar perturbación AC: amplitud 10 mV a 1 kHz
Pyvium.set_ac_frequency(1000.0)   # Hz
Pyvium.set_ac_amplitude(0.010)    # V (10 mV)
print("AC signal: 10 mV at 1 kHz")

Pyvium.set_potential(0.0)         # sesgo DC
Pyvium.set_cell_on()
time.sleep(0.5)

# Leer corriente resultante para una estimación aproximada de impedancia
I = Pyvium.get_current()
E = Pyvium.get_potential()
print(f"DC potential: {E:.4f} V | DC current: {I:.3e} A")

Pyvium.set_cell_off()

## 9. Selección de canal MUX

`set_mux_channel(n)` conmuta el multiplexor al canal dado (indexado desde 0). Úsalo cuando un accesorio multiplexor está conectado para rotar entre múltiples electrodos de trabajo.

In [ ]:
NUM_MUX_CHANNELS = 4
DWELL_TIME = 1.0  # segundos por canal

Pyvium.set_potential(0.0)
Pyvium.set_cell_on()

results = []
for ch in range(NUM_MUX_CHANNELS):
    Pyvium.set_mux_channel(ch)
    time.sleep(DWELL_TIME)
    I = Pyvium.get_current()
    results.append((ch, I))
    print(f"  MUX ch {ch}: I = {I:.3e} A")

Pyvium.set_cell_off()
Pyvium.set_mux_channel(0)  # restablecer al canal 0

## Limpieza

In [ ]:
Pyvium.set_cell_off()
Pyvium.disconnect_device()
Pyvium.close_driver()
print("Done")

---

## Resumen

| Capacidad | Método | Estado | Restricciones clave |
|-----------|--------|--------|-------------------|
| Ráfaga de corriente | `get_current_trace(n, interval)` | ⚠️ no funciona | Bloqueo de DLL — bajo investigación |
| Ráfaga de potencial | `get_potential_trace(n, interval)` | ⚠️ no funciona | Bloqueo de DLL — bajo investigación |
| Ráfaga de corriente WE2 | `get_current_we2_trace(n, interval)` | ⚠️ no funciona | Bloqueo de DLL — bajo investigación |
| Salida analógica | `set_dac(channel, volts)` | ✅ | canales 0 o 1 |
| Entrada analógica | `get_adc(channel)` | ✅ | canales 0–7 |
| Salida digital | `set_digital_output(bitmask)` | ✅ | bitmask entero |
| Entrada digital | `get_digital_input()` | ✅ | devuelve bitmask |
| Frecuencia AC | `set_ac_frequency(hz)` | ✅ | float en Hz |
| Amplitud AC | `set_ac_amplitude(volts)` | ✅ | float en V |
| Canal MUX | `set_mux_channel(n)` | ✅ | indexado desde 0; se requiere accesorio MUX |

## Siguiente

- **`05_bipotentiostat_and_we32.ipynb`** — Canal WE2 de BiStat y arreglo WE32 de 32 canales
- **`06_method_mode.ipynb`** — Ejecución completa de métodos electroquímicos (LSV, CV, EIS)